<a href="https://colab.research.google.com/github/seyda-sahin/ev-arkadasi-eslestirme-sistemi/blob/main/OOP_Proje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EV ARKADAŞI EŞLEŞTİRME SİSTEMİ

In [ ]:
import math
from uuid import UUID, uuid4
from datetime import datetime,timedelta
from enum import Enum
from decimal import Decimal
from typing import List, Dict
from __future__ import annotations

#-------------------------------------
# ENUM SINIFLARI
#-------------------------------------

class Cinsiyet(Enum):
  KADIN="Kadın"
  ERKEK="Erkek"

class Rol(Enum):
  OGRENCI="Öğrenci"
  EV_SAHIBI="Ev Sahibi"
  PASIF="Pasif"

class Kombi(Enum):
  MERKEZI="Merkezi"
  BIREYSEL="Bireysel"

class BasvuruDurumu(Enum):
  ASKIDA="Askıda"
  KABUL="Kabul"
  RED="Red"
  GERI_CEKILDI="Geri Çekildi"

class IlanDurumu(Enum):
  AKTIF="Aktif"
  PASIF="Pasif"
  KIRALANDI="Kiralandı"

class SozlesmeDurumu(Enum):
  TASLAK="Taslak"
  AKTIF="Aktif"
  SON_HAL="Son Hal"

#---------------------------------------
# Kullanıcı Sınıfı
#---------------------------------------

class Kullanici:
  def __init__(self,
               kullaniciId:UUID,
               kullaniciAdi:str,
               eposta:str,
               sifre:str,
               telefon:str,
               kayitTarihi:datetime,
               sonGiris:datetime,
               rol:Rol):

    self.kullaniciId=kullaniciId
    self.kullaniciAdi=kullaniciAdi
    self.eposta=eposta
    self.__sifre=sifre  #Private
    self.telefon=telefon
    self.kayitTarihi=kayitTarihi
    self.sonGiris=sonGiris
    self._rol=rol    #Protected

  def __str__(self):
        return f"""
        KullanıcıID:{self.kullaniciId},
        Kullanıcı Adı:{self.kullaniciAdi},
        Eposta:{self.eposta},
        Telefon:{self.telefon},
        Kayıt Tarihi:{self.kayitTarihi},
        Son Giriş:{self.sonGiris},
        Rol:{self._rol.name}"""


  def girisYap(self,sifre:str)->bool:
    if self.__sifre == sifre:
      print("Şifre doğru")
      return True
    else:
      print("Şifre yanlış")
      return False

  def sifreDegistir(self,eski:str,yeni:str)->bool:
    if self.__sifre==eski:
      self.__sifre=yeni
      return True
    return False

  def iletisimGuncelle(self,yeniEposta:str=None,yeniTelefon:str=None)->bool:
        degisti = False

        if yeniEposta is not None and yeniEposta != self.eposta:
            self.eposta = yeniEposta
            degisti = True

        if yeniTelefon is not None and yeniTelefon != self.telefon:
            self.telefon = yeniTelefon
            degisti = True

        return degisti

  def pasiflestirme(kullanicilar:list)->list:
    return [k for k in kullanicilar if k._rol==Rol.PASIF]

#---------------------------------------
# Öğrenci Sınıfı
#---------------------------------------

class Ogrenci(Kullanici):
  def __init__(self,
               kullaniciId:UUID,
               kullaniciAdi:str,
               eposta:str,
               sifre:str,
               telefon:str,
               kayitTarihi:datetime,
               sonGiris:datetime,
               rol:Rol,
               yas:int,
               cinsiyet:Cinsiyet,
               sigara:bool,
               evcilHayvanUygunluk:bool,
               temizlikSeviyesi:int,
               sosyalSeviye:int,
               butceMin:Decimal,
               butceMax:Decimal,
               tercihEdilenLokasyonlar:List[Lokasyon],
               profil:Profil,
               eslesmeler:List[Eslesme],
               tercihler:Tercihler):

    super().__init__(kullaniciId,
                     kullaniciAdi,
                     eposta,
                     sifre,
                     telefon,
                     kayitTarihi,
                     sonGiris,
                     rol)

    self._rol=Rol.OGRENCI   #Protected
    self.yas=yas
    self.cinsiyet=cinsiyet
    self.sigara=sigara
    self.evcilHayvanUygunluk=evcilHayvanUygunluk
    self.temizlikSeviyesi=temizlikSeviyesi
    self.sosyalSeviye=sosyalSeviye
    self.butceMin=butceMin
    self.butceMax=butceMax
    self.tercihEdilenLokasyonlar=tercihEdilenLokasyonlar
    self.profil=profil
    self._eslesmeler=eslesmeler    #Protected
    self.tercihler=tercihler
    self._basvurular:List[Basvuru]=[]     #Protected
    self._mesajlar:List[Mesaj]=[]         #Protected


    if not (1 <= temizlikSeviyesi <= 5):
      print("HATA:temizlikSeviyesi 1 ile 5 arasında olmalı")
    else:
      pass

    if not (1 <= sosyalSeviye <= 5):
      print("HATA:sosyalSeviye 1 ile 5 arasında olmalı")
    else:
      pass

  def tercihleriGuncelle(self,yeniTercihler:Tercihler):
      self._tercihler=yeniTercihler

  def eslesmeleriGoruntule(self)->List[Eslesme]:
    return self._eslesmeler

  def ilanaBasvur(self,ilan:Ilan)->Basvuru:
    yeni_basvuru=Basvuru(
        basvuruId=uuid4(),
        ilan=ilan,
        basvuran=self,
        mesaj="İlana başvuru yapıldı",
        durum=BasvuruDurumu.ASKIDA,
        basvuruTarihi=datetime.now()
    )
    self._basvurular.append(yeni_basvuru)
    return yeni_basvuru

  def mesajGonder(self,mesajId:UUID,aliciId:UUID,metin:str)->Mesaj:
    yeni_mesaj = Mesaj(
        mesajId=mesajId,
        gonderici=self.kullaniciId,
        alici=aliciId,
        metin=metin,
        gonderilmeTarihi=datetime.now(),
        okunmaTarihi=None
        )
    self._mesajlar.append(yeni_mesaj)
    return yeni_mesaj

  def degerlendirmeYap(self,eslesmeId:UUID,puan:int)->None:
        for eslesme in self._eslesmeler:
            if eslesme.eslesmeId == eslesmeId:
                eslesme.puanlar.append(puan)
                if sum(eslesme.puanlar)>7:
                    eslesme.eslesmeSaglandi = True
                return
        return

#-------------------------------------------
# Ev Sahibi Sınıfı
#-------------------------------------------

class EvSahibi(Kullanici):
  def __init__(self,
               kullaniciId:UUID,
               kullaniciAdi:str,
               eposta:str,
               sifre:str,
               telefon:str,
               kayitTarihi:datetime,
               sonGiris:datetime,
               rol:Rol,
               ilanlar:List[Ilan],
               dogrulama:bool):

    super().__init__(kullaniciId,
                     kullaniciAdi,
                     eposta,
                     sifre,
                     telefon,
                     kayitTarihi,
                     sonGiris,
                     rol)

    self._rol=Rol.EV_SAHIBI   #Protected
    self.ilanlar:List[Ilan]=[]
    self.dogrulama=dogrulama

  def ilanOlustur(self,
                  baslik:str,
                  aciklama:str,
                  fiyat:Decimal,
                  lokasyon:str,
                  maxKisi:int,
                  kurallar:str,
                  ev:Ev,
                  durum:IlanDurumu=IlanDurumu.AKTIF)->Ilan:

        yeni_ilan = Ilan(
        ilanId=uuid4(),
        sahibi=self,
        baslik=baslik,
        aciklama=aciklama,
        fiyat=fiyat,
        lokasyon=lokasyon,
        maxKisi=maxKisi,
        mevcutKiracilar=[],
        kurallar=kurallar,
        ev=ev,
        durum=durum
        )
        self.ilanlar.append(yeni_ilan)
        return yeni_ilan

  def ilanGuncelle(self,
                   ilanId:UUID,
                   baslik:str,
                   aciklama:str,
                   fiyat:Decimal,
                   lokasyon:str)->None:
                   for ilan in self.ilanlar:
                    if ilan.ilanId==ilanId:
                      ilan.baslik=baslik
                      ilan.aciklama=aciklama
                      ilan.fiyat=fiyat
                      ilan.lokasyon=lokasyon
                      return
                   return

  def basvurulariGoruntule(self,ilanId:UUID)->List[Basvuru]:
    for ilan in self.ilanlar:
      if ilan.ilanId==ilanId:
        return ilan.basvurular
    return

  def basvuruyuKabulEt(self,basvuruId:UUID)->Sozlesme:
    for ilan in self.ilanlar:
      for basvuru in ilan.basvurular:
        if basvuru.basvuruId==basvuruId:
          basvuru.kabulEdildi=True
          baslamaTarihi=datetime.now()
          bitisTarihi=baslamaTarihi + timedelta(days=365)   #Yani burada 1 yıllık sözleşme yaptım

          yeni_sozlesme=Sozlesme(
              sozlesmeId=uuid4(),
              ilan=ilan,
              kiraci=basvuru.basvuran,
              evSahibi=self,
              baslamaTarihi=baslamaTarihi,
              bitisTarihi=bitisTarihi,
              aylikKira=ilan.fiyat,
              depozito=ilan.fiyat * 2,
              durum=SozlesmeDurumu.TASLAK
              )
          return yeni_sozlesme
          return

  def basvuruyaMesajGonder(self,basvuruId:UUID,metin:str)->Mesaj:
    for ilan in self.ilanlar:
      for basvuru in ilan.basvurular:
        if basvuru.basvuruId==basvuruId:
          yeni_mesaj=Mesaj(
              mesajId=uuid4(),
              gonderici=self.kullaniciId,
              alici=basvuru.basvuran,
              metin=metin,
              gonderilmeTarihi=datetime.now(),
              okunmaTarihi=None
              )
          basvuru.mesajlar.append(yeni_mesaj)
          return yeni_mesaj
    return

#--------------------------------------------------
#  Mesaj Sınıfı
#--------------------------------------------------

class Mesaj:
  def __init__(self,
               mesajId:UUID,
               gonderici:Kullanici,
               alici:Kullanici,
               metin:str,
               gonderilmeTarihi:datetime,
               okunmaTarihi:datetime):

    self.mesajId=mesajId
    self.gonderici=gonderici
    self.alici=alici
    self.metin=metin
    self.gonderilmeTarihi=gonderilmeTarihi
    self.okunmaTarihi=okunmaTarihi

  def okunduOlarakIsaretle(self)->None:
    self.okunmaTarihi=datetime.now()

  def sil(self)->None:
    self.metin=None

#---------------------------------------------
# Profil Sınıfı
#---------------------------------------------

class Profil:
  def __init__(self,
               profil:UUID,
               hakkimda:str,
               egitimDurumu:str):

    self.profil=profil
    self.hakkimda=hakkimda
    self.egitimDurumu=egitimDurumu

  def hakkimdaGuncelle(self,yeniMetin:str)->None:
    self.hakkimda=yeniMetin

#-----------------------------------------------
# Tercihler Sınıfı
#-----------------------------------------------

class Tercihler:
  def __init__(self,
               maxKira:float,
               odaSayisiTercihi:int,
               tercihEdilenCinsiyet:Cinsiyet,
               evcilHayvanUygunlugu:bool,
               sigaraIzni:bool,
               minTemizlikSeviyesi:int,
               minSosyalSeviye:int):

    self.maxKira=maxKira
    self.odaSayisiTercihi=odaSayisiTercihi
    self.tercihEdilenCinsiyet=tercihEdilenCinsiyet
    self.evcilHayvanUygunlugu=evcilHayvanUygunlugu
    self.sigaraIzni=sigaraIzni
    self.minTemizlikSeviyesi=minTemizlikSeviyesi
    self.minSosyalSeviye=minSosyalSeviye

  def guncelle(self,maxKira:float)->None:
    self.maxKira=maxKira

  def eslesiyorMu(self,ilan:Ilan)->bool:
        if ilan.fiyat <= self.maxKira and ilan.ev.odaSayisi == self.odaSayisiTercihi:
            return True
        return False

#-------------------------------------------
# Başvuru Sınıfı
#-------------------------------------------

class Basvuru:
  def __init__(self,
               basvuruId:UUID,
               ilan:Ilan,
               basvuran:Ogrenci,
               mesaj:str,
               durum:BasvuruDurumu,
               basvuruTarihi:datetime):

    self.basvuruId=basvuruId
    self.ilan=ilan
    self.basvuran=basvuran
    self.mesaj=mesaj
    self.durum=durum
    self.basvuruTarihi=basvuruTarihi
    self.mesajlar:list[Mesaj]=[]

  def geriCek(self)->None:
    self.durum = BasvuruDurumu.GERI_CEKILDI

  def durumGuncelle(self,yeniDurum:BasvuruDurumu)->None:
    self.durum=yeniDurum

#----------------------------------------------
# İlan Sınıfı
#----------------------------------------------

class Ilan:
  def __init__(self,
               ilanId:UUID,
               sahibi:EvSahibi,
               baslik:str,
               aciklama:str,
               fiyat:Decimal,
               lokasyon:Lokasyon,
               maxKisi:int,
               mevcutKiracilar:List[Kullanici],
               kurallar:str,
               ev:Ev,
               durum:IlanDurumu):

    self.ilanId=ilanId
    self.sahibi=sahibi
    self.baslik=baslik
    self.aciklama=aciklama
    self.fiyat=fiyat
    self.lokasyon=lokasyon
    self.maxKisi=maxKisi
    self.mevcutKiracilar=mevcutKiracilar
    self.kurallar=kurallar
    self.ev=ev
    self.durum=durum

    self.basvurular:List[Basvuru]=[]

  def ilanaBasvur(self,ogr:Ogrenci,mesaj:str)->Basvuru:
        basvuruId=uuid4()
        basvuruTarihi=datetime.now()
        return Basvuru(basvuruId,self,ogr,mesaj,BasvuruDurumu.ASKIDA,basvuruTarihi)

  def eslesmePuaniHesapla(self,ogrenci:Ogrenci)->float:
    tercihler=ogrenci.tercihler
    puan=0.0

    if self.fiyat <= tercihler.maxKira:
      puan += 1.0
    if self.ev.odaSayisi == tercihler.odaSayisiTercihi:
      puan += 1.0
    if ogrenci.cinsiyet == tercihler.tercihEdilenCinsiyet:
      puan += 1.0
    if ogrenci.evcilHayvanUygunluk == tercihler.evcilHayvanUygunlugu:
      puan += 1.0
    if ogrenci.sigara == tercihler.sigaraIzni:
      puan += 1.0
    if ogrenci.temizlikSeviyesi >= tercihler.minTemizlikSeviyesi:
      puan += 1.0
    if ogrenci.sosyalSeviye >= tercihler.minSosyalSeviye:
      puan += 1.0
    if lokasyon in ogrenci.tercihEdilenLokasyonlar:
      puan += 1.0
      return puan

#------------------------------------
# Ev Sınıfı
#-----------------------------------

class Ev:
  def __init__(self,
               evId:UUID,
               odaSayisi:int,
               kat:int,
               metrekare:Decimal,
               kombiTipi:Kombi,
               evHakkindaAciklama:str):

    self.evId=evId
    self.odaSayisi=odaSayisi
    self.kat=kat
    self.metrekare=metrekare
    self.kombiTipi=kombiTipi
    self.evHakkindaAciklama=evHakkindaAciklama

  def evBilgisiGuncelle(self,yeniBilgi:Ev)->None:
    self.odaSayisi=yeniBilgi.odaSayisi
    self.kat=yeniBilgi.kat
    self.metrekare=yeniBilgi.metrekare
    self.kombiTipi=yeniBilgi.kombiTipi
    self.evHakkindaAciklama=yeniBilgi.evHakkindaAciklama

  def __str__(self):
        return f"Ev(odaSayisi={self.odaSayisi},kat={self.kat},metrekare={self.metrekare},kombiTipi={self.kombiTipi.name},aciklama={self.evHakkindaAciklama})"

#----------------------------------------
# Lokasyon Sınıfı
#----------------------------------------

class Lokasyon:
  def __init__(self,
               adress:str,
               sehir:str,
               ilce:str,
               enlem:Decimal,
               boylam:Decimal):

    self.adress=adress
    self.sehir=sehir
    self.ilce=ilce
    self.enlem=enlem
    self.boylam=boylam

  def mesafeHesapla(self,diger:Lokasyon)->float:   #İki mesafe arasındaki kuş uçuşu uzaklığı hesaplıyor
    dx=float(self.enlem) - float(diger.enlem)
    dy=float(self.boylam) - float(diger.boylam)
    return math.sqrt(dx**2 + dy**2)

  def yariCapIcindeMi(self,diger:Lokasyon,km:float)->bool:
    mesafe=self.mesafeHesapla(diger)
    return mesafe <= km

#------------------------------------
# Eşleşme Sınıfı
#------------------------------------

class Eslesme:
  def __init__(self,
               eslesmeId:UUID,
               ogrenci:Ogrenci,
               ilan:Ilan,
               puan:float,
               detaylar:Dict[str,float],
               olusturulmaTarihi:datetime):

    self.eslesmeId=eslesmeId
    self.ogrenci=ogrenci
    self.ilan=ilan
    self.puan=puan
    self.detaylar=detaylar
    self.olusturulmaTarihi=olusturulmaTarihi

    self.reddedildi=False
    self.puanlar:list[float]=[]
    self.eslesmeSaglandi=False


  def ogrenciKabul(self)->Basvuru:
    basvuru=Basvuru(
        basvuruId=uuid4(),
        ilan=self.ilan,
        basvuran=self.ogrenci,
        mesaj="Başvuru kabul edildi.",
        durum=BasvuruDurumu.KABUL,
        basvuruTarihi=datetime.now()
        )
    return basvuru

  def reddet(self)->None:
    self.reddedildi=True
    print(f"Eşleşme {self.eslesmeId} reddedildi.")

#-----------------------------------------
# Sözleşme Sınıfı
#-----------------------------------------

class Sozlesme:
  def __init__(self,
               sozlesmeId:UUID,
               ilan:Ilan,
               kiraci:Ogrenci,
               evSahibi:EvSahibi,
               baslamaTarihi:datetime,
               bitisTarihi:datetime,
               aylikKira:Decimal,
               depozito:Decimal,
               durum:SozlesmeDurumu):

    self.sozlesmeId=sozlesmeId
    self.ilan=ilan
    self.kiraci=kiraci
    self.evSahibi=evSahibi
    self.baslamaTarihi=baslamaTarihi
    self.bitisTarihi=bitisTarihi
    self.aylikKira=aylikKira
    self.depozito=depozito
    self.durum=durum

    self.imzalayan=None
    self.iptalSebebi=None

  def imzala(self,kullanici:Kullanici)->None:
    self.imzalayan=kullanici
    self.durum=SozlesmeDurumu.AKTIF
    print(f"Sözleşme {self.sozlesmeId} {kullanici.kullaniciAdi} tarafından imzalandı.")

  def iptalEt(self,sebep:str)->None:
    self.durum=SozlesmeDurumu.SON_HAL
    self.iptalSebebi=sebep
    print(f"Sözleşme {self.sozlesmeId} iptal edildi.Sebep:{sebep}.Durum:{self.durum.value}")


Test Kodları

In [ ]:
#Kullanıcı Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

k1=Kullanici(uuid4(),"seyda","seyda@mail.com","1234","05368739298",datetime.now(),datetime.now(),Rol.OGRENCI)

k2=Kullanici(uuid4(),"verda","verda@mail.com","abcd","05123446754",datetime.now(),datetime.now(),Rol.PASIF)

#Giriş yap metodu
print("\n-> Giriş Testi <-")
k1.girisYap("1234")   #doğru şifre girdim
k1.girisYap("0000")   #yanlış şifre girdim

#Şifre değiştirme metodu
print("\n-> Şifre Değiştirme Testi <-")
k1.sifreDegistir("1234","yeni123")
k1.girisYap("yeni123")  #yeni şifreyle giriş yapma

#İletişim bilgilerini güncelleme metodu
print("\n-> İletişim Güncelleme Testi <-")
k1.iletisimGuncelle(yeniEposta="seyda_yeni@gmail.com")
k1.iletisimGuncelle(yeniTelefon="05874443210")
print(k1)

#Pasifleştirme metodu
print("\n-> Pasifleştirme Testi <-")
pasifler=Kullanici.pasiflestirme([k1, k2])
for p in pasifler:
    print("Pasif kullanıcı:",p.kullaniciAdi)


-> Giriş Testi <-
Şifre doğru
Şifre yanlış

-> Şifre Değiştirme Testi <-
Şifre doğru

-> İletişim Güncelleme Testi <-

        KullanıcıID:38d90331-262d-4239-9141-2680db6206c5,
        Kullanıcı Adı:seyda,
        Eposta:seyda_yeni@gmail.com,
        Telefon:05874443210,
        Kayıt Tarihi:2025-12-23 06:24:27.221221,
        Son Giriş:2025-12-23 06:24:27.221223,
        Rol:OGRENCI

-> Pasifleştirme Testi <-
Pasif kullanıcı: verda


In [ ]:
#Öğrenci Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

ogr = Ogrenci(
    kullaniciId=uuid4(),
    kullaniciAdi="seyda",
    eposta="seyda@gmail.com",
    sifre="1234",
    telefon="05871234445",
    kayitTarihi=datetime.now(),
    sonGiris=datetime.now(),
    rol=Rol.OGRENCI,
    yas=22,
    cinsiyet=Cinsiyet.KADIN,
    sigara=False,
    evcilHayvanUygunluk=True,
    temizlikSeviyesi=3,
    sosyalSeviye=4,
    butceMin=Decimal("2000"),
    butceMax=Decimal("4000"),
    tercihEdilenLokasyonlar=[
        Lokasyon(sehir="Malatya",ilce="Battalgazi",enlem=38.355,boylam=38.333,adress="Aşağıbağlar Mahallesi,GaziBaba Sokak")],
    profil=Profil(
        profil="Üniversite öğrencisi",
        hakkimda="Malatya'da okuyorum,ev arkadaşı arıyorum.",
        egitimDurumu="Lisans-Bilgisayar Mühendisliği"),
    eslesmeler=[
        Eslesme(
            eslesmeId=uuid4(),
            ogrenci="seyda",
            ilan="ilan-123",
            puan=0,
            detaylar="-",
            olusturulmaTarihi=datetime.now())],
    tercihler=Tercihler(
    maxKira=5000.0,
    odaSayisiTercihi=2,
    tercihEdilenCinsiyet=Cinsiyet.KADIN,
    evcilHayvanUygunlugu=True,
    sigaraIzni=False,
    minTemizlikSeviyesi=4,
    minSosyalSeviye=3)
)

#Tercihleri güncelleme metodu
yeni_tercih=Tercihler(
    maxKira=4500.0,
    odaSayisiTercihi=1,
    tercihEdilenCinsiyet=Cinsiyet.KADIN,
    evcilHayvanUygunlugu=False,
    sigaraIzni=False,
    minTemizlikSeviyesi=5,
    minSosyalSeviye=2
)

ogr.tercihleriGuncelle(yeni_tercih)
print("Yeni tercih güncellendi:", ogr._tercihler.maxKira)

#Eşleşmeleri görüntüleme metodu
print("Eşleşmeler:", ogr.eslesmeleriGoruntule())

#İlana başvuru metodu
basvuru = ogr.ilanaBasvur(uuid4())
print("Yeni başvuru:", basvuru)

#Mesaj gönderme metodu
mesaj = ogr.mesajGonder(uuid4(), uuid4(), "Merhaba, ev arkadaşı arıyor musun?")
print("Yeni mesaj:", mesaj.metin)

#Değerlendirme yapma metodu
eslesmeId = ogr._eslesmeler[0].eslesmeId
ogr.degerlendirmeYap(eslesmeId,5)   #Öğrenci eşleşmeye 5 puan verdi
#ogr.degerlendirmeYap(eslesmeId,4)
if ogr._eslesmeler[0].eslesmeSaglandi:
    print("Eşleşme sağlandı")
else:
    print("Henüz eşleşme olmadı")

print("Eşleşme puanları:",ogr._eslesmeler[0].puanlar)

Yeni tercih güncellendi: 4500.0
Eşleşmeler: [<__main__.Eslesme object at 0x7d02bc2767b0>]
Yeni başvuru: <__main__.Basvuru object at 0x7d02bc2769f0>
Yeni mesaj: Merhaba, ev arkadaşı arıyor musun?
Henüz eşleşme olmadı
Eşleşme puanları: [5]


In [ ]:
#Ev Sahibi Sınıfı Kontrol
from uuid import uuid4
from datetime import datetime
from decimal import Decimal

#Ev sahibi oluşturma
ev_sahibi=EvSahibi(
    kullaniciId=uuid4(),
    kullaniciAdi="ayse",
    eposta="ayse@gmail.com",
    sifre="12345",
    telefon="05012345647",
    kayitTarihi=datetime(2024,5,1),
    sonGiris=datetime(2025,1,1),
    rol=Rol.EV_SAHIBI,
    ilanlar=[],
    dogrulama=True
)

#Ev sahibinin ilana koyduğu evi oluşturma
ev=Ev(
    evId=uuid4(),
    odaSayisi=3,
    kat=2,
    metrekare=Decimal("120"),
    kombiTipi="Merkezi",
    evHakkindaAciklama="Üniversiteye yakın ve güney cephe."
)

#İlan oluşturma metodu
ilan=ev_sahibi.ilanOlustur(
    baslik="Merkezde 2+1 Daire",
    aciklama="Üniversiteye yakın, geniş ve ferah daire.",
    fiyat=Decimal("7500.00"),
    lokasyon="Malatya",
    maxKisi=3,
    kurallar="Sigara içilemez,evcil hayvan yasak",
    ev=ev
)
print("İlan oluşturuldu:",ilan.baslik,ilan.fiyat)

#İlan güncelleme metodu
ev_sahibi.ilanGuncelle(
    ilanId=ilan.ilanId,
    baslik="Yeni yapılmış 2+1 Daire",
    aciklama="Üniversiteye yakın ve yeni boyandı",
    fiyat=Decimal("8000.00"),
    lokasyon="Malatya Merkez"
)
print("İlan güncellendi:",ilan.baslik, ilan.fiyat)

#Örnek kiracı oluşturma
kiraci = Ogrenci(
    kullaniciId=uuid4(),
    kullaniciAdi="fatma",
    eposta="fatma@gmail.com",
    sifre="abcde",
    telefon="05765432134",
    kayitTarihi=datetime(2024,6,15),
    sonGiris=datetime(2025,1,2),
    rol=Rol.OGRENCI,
    yas=22,
    cinsiyet="Erkek",
    sigara=False,
    evcilHayvanUygunluk=True,
    temizlikSeviyesi=4,
    sosyalSeviye=3,
    butceMin=Decimal("5000.00"),
    butceMax=Decimal("8000.00"),
    tercihEdilenLokasyonlar=["Malatya Merkez","Battalgazi","Yesilyurt"],
    profil="Üniversite öğrencisiyim,sakin biriyim",
    eslesmeler=[],
    tercihler={"odaSayisi":2,"kat":2}
)

#Başvuru görüntüleme metodu
basvuru=Basvuru(
    basvuruId=uuid4(),
    ilan=ilan,
    basvuran=kiraci,
    mesaj="Bu ev ile ilgileniyorum.",
    durum=BasvuruDurumu.ASKIDA,
    basvuruTarihi=datetime.now()
)
ilan.basvurular.append(basvuru)
print("Başvuru eklendi:",basvuru.basvuruId,"Basvuran:",basvuru.basvuran.kullaniciAdi)

#Başvuruyu kabul etme metodu
sozlesme=ev_sahibi.basvuruyuKabulEt(basvuruId=basvuru.basvuruId)
print("Sözleşme oluşturuldu:",sozlesme.sozlesmeId,
      "Kiracı:",sozlesme.kiraci.kullaniciAdi,
      "Aylık kira:",sozlesme.aylikKira)

#Başvuruya mesaj gönderme metodu
mesaj=ev_sahibi.basvuruyaMesajGonder(
    basvuruId=basvuru.basvuruId,
    metin="Başvurunuz kabul edilmiştir."
)
print("Mesaj gönderildi:",mesaj.metin,"=>",mesaj.alici.kullaniciAdi)

İlan oluşturuldu: Merkezde 2+1 Daire 7500.00
İlan güncellendi: Yeni yapılmış 2+1 Daire 8000.00
Başvuru eklendi: 9c1e97d5-c3c6-4e5a-8578-2fbd1c03a2a9 Basvuran: fatma
Sözleşme oluşturuldu: 366cd656-0729-41bc-b741-7922f8c030c0 Kiracı: fatma Aylık kira: 8000.00
Mesaj gönderildi: Başvurunuz kabul edilmiştir. => fatma


In [ ]:
#Mesaj Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

gonderici=Kullanici(
    kullaniciId=uuid4(),
    kullaniciAdi="Ev Sahibi",
    eposta="evsahibi@gmail.com",
    sifre="1234",
    telefon="5551112233",
    kayitTarihi=datetime.now(),
    sonGiris=datetime.now(),
    rol="EV_SAHIBI"
)

alici = Kullanici(
    kullaniciId=uuid4(),
    kullaniciAdi="Öğrenci",
    eposta="ogrenci@gmail.com",
    sifre="abcd",
    telefon="5554443322",
    kayitTarihi=datetime.now(),
    sonGiris=datetime.now(),
    rol="OGRENCI"
)

mesaj = Mesaj(
    mesajId=uuid4(),
    gonderici=gonderici,
    alici=alici,
    metin="Merhaba,başvurunuz incelendi.",
    gonderilmeTarihi=datetime.now(),
    okunmaTarihi=None
)

#Mesaj oluşturma
print("Mesaj oluşturuldu:",mesaj.metin)

#Mesajı okundu olarak işaretleme metodu
mesaj.okunduOlarakIsaretle()
print("Mesaj okundu tarihi:",mesaj.okunmaTarihi)

#Mesaj silme metodu
mesaj.sil()
print("Mesaj silindikten sonra metin:",mesaj.metin)

Mesaj oluşturuldu: Merhaba,başvurunuz incelendi.
Mesaj okundu tarihi: 2025-12-23 06:25:59.639129
Mesaj silindikten sonra metin: None


In [ ]:
#Profil Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

profil = Profil(
    profil=uuid4(),
    hakkimda="Merhaba,ben şeyda.",
    egitimDurumu="Üniversite Öğrencisi"
)

print("Başlangıçtaki hakkımda bilgisi:", profil.hakkimda)

# Hakkımda kısmı güncelleme metodu
profil.hakkimdaGuncelle("Merhaba,ben şeyda şahin.")
print("Güncellenmiş hakkımda bilgisi:", profil.hakkimda)

Başlangıçtaki hakkımda bilgisi: Merhaba,ben şeyda.
Güncellenmiş hakkımda bilgisi: Merhaba,ben şeyda şahin.


In [ ]:
#Tercihler Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

tercih=Tercihler(
    maxKira=3000,
    odaSayisiTercihi=2,
    tercihEdilenCinsiyet=Cinsiyet.KADIN,
    evcilHayvanUygunlugu=True,
    sigaraIzni=False,
    minTemizlikSeviyesi=3,
    minSosyalSeviye=2
)

#Eşleşiyor mu metodu için komtroller
#Ev nesneleri oluşturma
ev1 = Ev(uuid4(),odaSayisi=2,kat=3,metrekare=120,kombiTipi="Merkezi",evHakkindaAciklama="Geniş ve ferah daire")
ev2 = Ev(uuid4(),odaSayisi=2,kat=1,metrekare=90,kombiTipi="Bireysel",evHakkindaAciklama="Merkezi konumda")
ev3 = Ev(uuid4(),odaSayisi=3,kat=2,metrekare=100,kombiTipi="Bireysel",evHakkindaAciklama="Sessiz ve huzurlu apartman")

#İlan nesneleri oluşturma
ilan1 = Ilan(uuid4(),"Ev Sahibi","Merkezi ev","Ferah daire",2500,"Malatya",2,[],"Sigara içilmez",ev1,"AKTIF")
ilan2 = Ilan(uuid4(),"Ev Sahibi","Uygun ev","Merkezi konum",3500,"Malatya",2,[],"Evcil hayvan kabul edilir",ev2,"AKTIF")
ilan3 = Ilan(uuid4(),"Ev Sahibi","Sessiz ev","Huzurlu ortam",2000,"Malatya",3,[],"Sigara içilmez",ev3,"AKTIF")

#Çıktılar
print("İlan1 eşleşiyor mu?", tercih.eslesiyorMu(ilan1))  # True
print("İlan2 eşleşiyor mu?", tercih.eslesiyorMu(ilan2))  # False (fiyat yüksek olduğundan dolayı)
print("İlan3 eşleşiyor mu?", tercih.eslesiyorMu(ilan3))  # False (oda sayısı farklı olduğu için)

#Güncelleme metodu
tercih.guncelle(4000)
print("Yeni maxKira:",tercih.maxKira)
print("İlan2 eşleşiyor mu (güncellemeden sonra)?",tercih.eslesiyorMu(ilan2))  # True

İlan1 eşleşiyor mu? True
İlan2 eşleşiyor mu? False
İlan3 eşleşiyor mu? False
Yeni maxKira: 4000
İlan2 eşleşiyor mu (güncellemeden sonra)? True


In [ ]:
#Başvuru Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

profil=Profil(uuid4(),"Merhaba ben Şeyda","Üniversite öğrencisi")
tercihler=Tercihler(3000,2,Cinsiyet.KADIN,True,False,3,2)

ogrenci = Ogrenci(
    kullaniciId=uuid4(),
    kullaniciAdi="Şeyda",
    eposta="seyda@gmail.com",
    sifre="1234",
    telefon="05678342134",
    kayitTarihi=datetime.now(),
    sonGiris=datetime.now(),
    rol=Rol.OGRENCI,
    yas=22,
    cinsiyet=Cinsiyet.KADIN,
    sigara=False,
    evcilHayvanUygunluk=True,
    temizlikSeviyesi=4,
    sosyalSeviye=3,
    butceMin=2000,
    butceMax=3500,
    tercihEdilenLokasyonlar=["Malatya", "İstanbul"],
    profil=profil,
    eslesmeler=[],
    tercihler=tercihler
)

ev=Ev(uuid4(),2,3,120,"Bireysel","Geniş ve ferah daire")
ilan=Ilan(uuid4(),"Ev Sahibi","Merkezi Ev","Ferah daire",2500,"Malatya",2,[],"Sigara içilmez",ev,"AKTIF")

basvuru=Basvuru(
    basvuruId=uuid4(),
    ilan=ilan,
    basvuran=ogrenci,
    mesaj="Ev için başvuru yapmak istiyorum.",
    durum=BasvuruDurumu.ASKIDA,
    basvuruTarihi=datetime.now()
)

#Başvuru durumunu güncelleme metodu
print("Başlangıç durumu:",basvuru.durum)
basvuru.durumGuncelle(BasvuruDurumu.KABUL)
print("Durum güncellendi:",basvuru.durum)

#Başvuruyu geri çekme metodu
basvuru.geriCek()
print("Başvuru geri çekildi, yeni durum:",basvuru.durum)

Başlangıç durumu: BasvuruDurumu.ASKIDA
Durum güncellendi: BasvuruDurumu.KABUL
Başvuru geri çekildi, yeni durum: BasvuruDurumu.GERI_CEKILDI


In [ ]:
#İlan Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

ev=Ev(uuid4(),2,3,Decimal("120"),Kombi.MERKEZI,"Geniş ve ferah daire")

lokasyon=Lokasyon("Zafer Mah.","Malatya","Yeşilyurt",Decimal("38.355"),Decimal("38.333"))

sahip=EvSahibi(uuid4(),"Ev Sahibi","evsahibi@gmail.com","1234","05698761243",datetime.now(),datetime.now(),Rol.EV_SAHIBI,[],True)

tercihler=Tercihler(3000,2,Cinsiyet.KADIN,True,False,3,2)

ogrenci=Ogrenci(uuid4(),"Şeyda","seyda@gmail.com","abcd","05467651928",datetime.now(),datetime.now(),Rol.OGRENCI,22,Cinsiyet.KADIN,False,True,4,3,2000,3500,["Malatya","İstanbul"],None,[],tercihler)

ilan=Ilan(uuid4(),sahip,"Merkezi Ev","Ferah daire",Decimal("2500"),lokasyon,2,[],"Sigara içilmez",ev,IlanDurumu.AKTIF)

#İlana başvurma metodu
basvuru=ilan.ilanaBasvur(ogrenci,"Ev için başvuru yapmak istiyorum.")
print("Başvuru oluşturuldu,durum:",basvuru.durum)

#Eşleşme puanı hesaplama
puan=ilan.eslesmePuaniHesapla(ogrenci)
print("Eşleşme puanı:",puan)


Başvuru oluşturuldu,durum: BasvuruDurumu.ASKIDA
Eşleşme puanı: None


In [ ]:
#Ev Sınıfı Kontrol
from uuid import uuid4, UUID
from datetime import datetime
from enum import Enum

#Ev nesnesi oluşturma
ev1=Ev(uuid4(),3,2,Decimal("120.5"),Kombi.MERKEZI,"Şehir merkezinde güzel bir daire")
print("İlk ev bilgisi:",ev1)

#Yeni ev nesnesi oluşturma
ev2=Ev(uuid4(),4,5,Decimal("150.0"),Kombi.BIREYSEL,"Manzaralı geniş daire")

#Ev bilgisi güncelleme
ev1.evBilgisiGuncelle(ev2)
print("Güncellenmiş ev bilgisi:",ev1)

İlk ev bilgisi: Ev(odaSayisi=3,kat=2,metrekare=120.5,kombiTipi=MERKEZI,aciklama=Şehir merkezinde güzel bir daire)
Güncellenmiş ev bilgisi: Ev(odaSayisi=4,kat=5,metrekare=150.0,kombiTipi=BIREYSEL,aciklama=Manzaralı geniş daire)


In [ ]:
#Lokasyon Sınıfı Kontrol
from decimal import Decimal
import math

lok1=Lokasyon("Çiçek Sok.","Malatya","Yeşilyurt",Decimal("38.35"),Decimal("38.30"))
lok2=Lokasyon("Gül Sok.","Malatya","Battalgazi",Decimal("38.36"),Decimal("38.32"))

print("Lokasyon 1:",lok1.adress,lok1.sehir,lok1.ilce)
print("Lokasyon 2:",lok2.adress,lok2.sehir,lok2.ilce)

#Mesafe hesaplama metodu
mesafe=lok1.mesafeHesapla(lok2)
print("İki lokasyon arasındaki mesafe:",mesafe)

#Yarıçap kontrolü metodu
print("5 km içinde mi?",lok1.yariCapIcindeMi(lok2,5))
print("0.01 km içinde mi?",lok1.yariCapIcindeMi(lok2,0.01))


Lokasyon 1: Çiçek Sok. Malatya Yeşilyurt
Lokasyon 2: Gül Sok. Malatya Battalgazi
İki lokasyon arasındaki mesafe: 0.022360679774999805
5 km içinde mi? True
0.01 km içinde mi? False


In [ ]:
#Eşleşme Sınıfı Kontrol
#Öğrenci nesnesi oluşturma
ogrenci=Ogrenci(
    kullaniciId=uuid4(),
    kullaniciAdi="selin",
    eposta="selin@gamil.com",
    sifre="12345",
    telefon="05467891536",
    kayitTarihi=datetime.now(),
    sonGiris=datetime.now(),
    rol="ogrenci",
    yas=22,
    cinsiyet="Erkek",
    sigara=False,
    evcilHayvanUygunluk=True,
    temizlikSeviyesi=4,
    sosyalSeviye=3,
    butceMin=2000,
    butceMax=4000,
    tercihEdilenLokasyonlar=["Malatya","Elazığ"],
    profil="Bilgisayar mühendisliği öğrencisi",
    eslesmeler=[],
    tercihler={}
)

#Ev nesnesi oluşturma
ev=Ev(
    evId=uuid4(),
    odaSayisi=3,
    kat=2,
    metrekare=Decimal("120.5"),
    kombiTipi=Kombi("Merkezi"),
    evHakkindaAciklama="Yüksek,manzaralı,merkezi konumda."
)

#İlan nesnesi oluşturma
ilan=Ilan(
    ilanId=uuid4(),
    baslik="Battalgazi'de ev arkadaşı aranıyor",
    aciklama="3+1 dairede bir oda boş.Evcil hayvan kabul edilir,sigara içilmez.",
    sahibi=ogrenci,
    fiyat=3000,
    lokasyon="Sİvas",
    maxKisi=2,
    mevcutKiracilar=[],
    kurallar=["Sigara içilmez","Evcil hayvan kabul edilir"],
    ev=ev,
    durum="aktif"
)

#Eşleşme nesnesi oluşturma
eslesme=Eslesme(
    eslesmeId=uuid4(),
    ogrenci=ogrenci,
    ilan=ilan,
    puan=4.5,
    detaylar={
        "temizlik":2.0,
        "sosyallik":1.5,
        "bütçeUyumu":1.0},
    olusturulmaTarihi=datetime.now()
)

print("Eşleşme oluşturuldu:",eslesme.eslesmeId,"- Puan:",eslesme.puan)
print("Detaylar:",eslesme.detaylar)

#Kabul etme metodu
basvuru=eslesme.ogrenciKabul()
print("Basvuru ID:",basvuru.basvuruId)
print("Basvuran:",basvuru.basvuran.kullaniciAdi)
print("İlan:",basvuru.ilan.baslik)
print("Mesaj:",basvuru.mesaj)
print("Durum:",basvuru.durum)

#Reddetme metodu
eslesme.reddet()
print("Reddedildi mi?",eslesme.reddedildi)

Eşleşme oluşturuldu: e83dbc18-c52b-434e-8ba5-0b7fc1889792 - Puan: 4.5
Detaylar: {'temizlik': 2.0, 'sosyallik': 1.5, 'bütçeUyumu': 1.0}
Basvuru ID: c04d54b8-8942-480c-b28f-f45584bed7d1
Basvuran: selin
İlan: Battalgazi'de ev arkadaşı aranıyor
Mesaj: Başvuru kabul edildi.
Durum: BasvuruDurumu.KABUL
Eşleşme e83dbc18-c52b-434e-8ba5-0b7fc1889792 reddedildi.
Reddedildi mi? True


In [ ]:
#Sözleşme Sınıfı Kontrol
import uuid
from datetime import datetime
from decimal import Decimal
from enum import Enum

#Ev sahibi nesnesi oluşturma
ev_sahibi=EvSahibi(
    kullaniciId=uuid.uuid4(),
    kullaniciAdi="ahmet",
    eposta="ahmet@gmail.com",
    sifre="12345",
    telefon="05045238976",
    kayitTarihi=datetime(2024,5,1),
    sonGiris=datetime(2025,1,1),
    rol=Rol.EV_SAHIBI,
    ilanlar=[],
    dogrulama=True
)

#Ev nesnesi oluşturma
ev=Ev(
    evId=uuid.uuid4(),
    odaSayisi=3,
    kat=2,
    metrekare=Decimal("120"),
    kombiTipi="Merkezi",
    evHakkindaAciklama="Üniversiteye yakın,ferah ve güneş alan daire."
)

#İlan nesnesi oluşturma
ilan=Ilan(
    ilanId=uuid.uuid4(),
    sahibi=ev_sahibi,
    baslik="Merkezde 2+1 Daire",
    aciklama="Üniversiteye yakın,geniş ve ferah daire.",
    fiyat=Decimal("7500.00"),
    lokasyon="Malatya",
    maxKisi=3,
    mevcutKiracilar=[],
    kurallar=["Sigara içilmez","Evcil hayvan yok"],
    ev=ev,
    durum=IlanDurumu.AKTIF
)

#Kiracı nesnesi oluşturma
kiraci=Ogrenci(
    kullaniciId=uuid.uuid4(),
    kullaniciAdi="zafer",
    eposta="zafer@gmail.com",
    sifre="abcde",
    telefon="05027654321",
    kayitTarihi=datetime(2024,6,15),
    sonGiris=datetime(2025,1,2),
    rol=Rol.OGRENCI,
    yas=22,
    cinsiyet="Erkek",
    sigara=False,
    evcilHayvanUygunluk=True,
    temizlikSeviyesi=4,
    sosyalSeviye=3,
    butceMin=Decimal("5000.00"),
    butceMax=Decimal("8000.00"),
    tercihEdilenLokasyonlar=["Malatya Merkez","Battalgazi"],
    profil="Üniversite öğrencisi,sakin yaşam tarzı.",
    eslesmeler=[],
    tercihler={"odaSayisi":2,"kat":2}
)

#Sözleşme oluşturma
sozlesme=Sozlesme(
    sozlesmeId=uuid.uuid4(),
    ilan=ilan,
    kiraci=kiraci,
    evSahibi=ev_sahibi,
    baslamaTarihi=datetime(2025, 1, 1),
    bitisTarihi=datetime(2025, 12, 31),
    aylikKira=Decimal("7500.00"),
    depozito=Decimal("15000.00"),
    durum=SozlesmeDurumu.TASLAK
)

print("Sözleşme başlangıç durumu")
print("ID:",sozlesme.sozlesmeId)
print("İlan:",sozlesme.ilan)
print("Kiracı:",sozlesme.kiraci)
print("Ev Sahibi:",sozlesme.evSahibi)
print("Durum:",sozlesme.durum.value)

#İmzalama metodu
kullanici = Kullanici(
    kullaniciId=uuid.uuid4(),
    kullaniciAdi="hafize",
    eposta="hafize@gmail.com",
    sifre="abcde",
    telefon="05567654321",
    kayitTarihi=datetime(2024,6,15),
    sonGiris=datetime(2025,1,2),
    rol=Rol.OGRENCI
)
sozlesme.imzala(kullanici)

#İptal etme metodu
sozlesme.iptalEt("Kiracı taşınmaktan vazgeçti")
print("Sözleşme son durumu")
print("Durum:",sozlesme.durum.value)
print("İptal sebebi:",sozlesme.iptalSebebi)

Sözleşme başlangıç durumu
ID: 18ef6a0e-8de3-4607-b69d-87ebf26b7192
İlan: <__main__.Ilan object at 0x7b52fea33110>
Kiracı: 
        KullanıcıID:3601c744-d3a0-4f99-b7b3-705f4b115d29,
        Kullanıcı Adı:zafer,
        Eposta:zafer@gmail.com,
        Telefon:05027654321,
        Kayıt Tarihi:2024-06-15 00:00:00,
        Son Giriş:2025-01-02 00:00:00,
        Rol:OGRENCI
Ev Sahibi: 
        KullanıcıID:0629bcf0-3122-4a48-829a-2d99ce3f35aa,
        Kullanıcı Adı:ahmet,
        Eposta:ahmet@gmail.com,
        Telefon:05045238976,
        Kayıt Tarihi:2024-05-01 00:00:00,
        Son Giriş:2025-01-01 00:00:00,
        Rol:EV_SAHIBI
Durum: Taslak
Sözleşme 18ef6a0e-8de3-4607-b69d-87ebf26b7192 hafize tarafından imzalandı.
Sözleşme 18ef6a0e-8de3-4607-b69d-87ebf26b7192 iptal edildi.Sebep:Kiracı taşınmaktan vazgeçti.Durum:Son Hal
Sözleşme son durumu
Durum: Son Hal
İptal sebebi: Kiracı taşınmaktan vazgeçti


In [ ]:
#Terminalden inputlarla giriş kod örneği

print("Ev Arkadaşı Eşleştirme Sistemi")

#Öğrenci bilgileri girişi
ad=input("Öğrenci adı: ")
butce_max=Decimal(input("Maksimum bütçe: "))
temizlik_seviyesi=int(input("Öğrencinin temizlik seviyesi (1-5): "))
sosyallik_seviyesi=int(input("Öğrencinin sosyallik seviyesi (1-5): "))

#Tercihler girişi
oda_tercihi=int(input("Tercih edilen oda sayısı: "))
min_temizlik=int(input("Minimum temizlik tercihi (1-5): "))
min_sosyallik=int(input("Minimum sosyallik tercihi (1-5): "))


#Ev
ev=Ev(
    evId=uuid4(),
    odaSayisi=3,
    kat=2,
    metrekare=Decimal("120"),
    kombiTipi=Kombi.BIREYSEL,
    evHakkindaAciklama="Güney cephede"
)

#Lokasyon
lokasyon=Lokasyon(
    adress="Merkez",
    sehir="Malatya",
    ilce="Battalgazi",
    enlem=Decimal("38.355"),
    boylam=Decimal("38.309")
)

#Tercihler => inputlar buraya etki edecek
tercihler=Tercihler(
    maxKira=butce_max,
    odaSayisiTercihi=3,
    tercihEdilenCinsiyet=Cinsiyet.KADIN,
    evcilHayvanUygunlugu=False,
    sigaraIzni=False,
    minTemizlikSeviyesi=min_temizlik,
    minSosyalSeviye=min_sosyallik
)

#Profil
profil=Profil(
    profil=uuid4(),
    hakkimda="Ben üniversite okuyorum",
    egitimDurumu="Bilgisayar Mühendisliği"
)

#Öğrenci
ogrenci=Ogrenci(
    kullaniciId=uuid4(),
    kullaniciAdi=ad,
    eposta="busra@gmail.com",
    sifre="1234",
    telefon="05564335421",
    kayitTarihi=datetime.now(),
    sonGiris=datetime.now(),
    rol=Rol.OGRENCI,
    yas=21,
    cinsiyet=Cinsiyet.KADIN,
    sigara=False,
    evcilHayvanUygunluk=False,
    temizlikSeviyesi=temizlik_seviyesi,
    sosyalSeviye=sosyallik_seviyesi,
    butceMin=Decimal("5000"),
    butceMax=butce_max,
    tercihEdilenLokasyonlar=[lokasyon],
    profil=profil,
    eslesmeler=[],
    tercihler=tercihler
)

#Ev sahibi
ev_sahibi=EvSahibi(
    kullaniciId=uuid4(),
    kullaniciAdi="Ev Sahibi",
    eposta="ayse@gmail.com",
    sifre="4321",
    telefon="05554987659",
    kayitTarihi=datetime.now(),
    sonGiris=datetime.now(),
    rol=Rol.EV_SAHIBI,
    ilanlar=[],
    dogrulama=True
)

ilan=ev_sahibi.ilanOlustur(
    baslik="3+1 Ev",
    aciklama="Üniversiteye yakın",
    fiyat=Decimal("10000"),
    lokasyon=lokasyon,
    maxKisi=2,
    kurallar="Sigara yasak,evcil hayvan yasak",
    ev=ev
)

puan=ilan.eslesmePuaniHesapla(ogrenci)

print("\n-SONUÇ-")
print(f"Eşleşme puanı: {puan}")

if puan >= 7:
    print("Eşleşme sağlandı.")
else:
    print("Eşleşme sağlanamadı.")


#odaSayisiTercihi=3 -> odaSayisi=3                                                                 => +1 puan
#tercihEdilenCinsiyet == ogrenci.cinsiyet(KADIN)                                                   => +1 puan
#evcilHayvanUygunlugu == ogrenci.evcilHayvanUygunluk(False)                                        => +1 puan
#sigaraIzni == ogrenci.sigara(False)                                                               => +1 puan
#lokasyon in tercihEdilenLokasyonlar                                                               => +1 puan
#maxKira >= ilan.fiyat(10000) ->  butce_max >= ilan.fiyat(10000)                                   => +1 puan
#minTemizlikSeviyesi <= ogrenci.temizlikSeviyesi -> min_temizlik <= temizlik_seviyesi              => +1 puan
#minSosyalSeviye <= ogrenci.sosyalSeviye -> min_sosyal <= sosyal_seviyesi                          => +1 puan

Ev Arkadaşı Eşleştirme Sistemi
Öğrenci adı: busra
Maksimum bütçe: 12000
Öğrencinin temizlik seviyesi (1-5): 4
Öğrencinin sosyallik seviyesi (1-5): 3
Tercih edilen oda sayısı: 3
Minimum temizlik tercihi (1-5): 3
Minimum sosyallik tercihi (1-5): 3

-SONUÇ-
Eşleşme puanı: 8.0
Eşleşme sağlandı.
